# `integrate_results.csv`

Connor Shrader / 09-21-2025

This file loads in and computes metrics on all of the clone results and saves to the file `all_clone_results.csv`.

In [1]:
import os
import pandas as pd
import numpy as np
import ast
import testis_model as tm
from tqdm import tqdm

from saving_loading import find_scenarios, load_all_scenarios, load_clone_names
from testis_model.metrics import init_time, last_time, min_time, max_time, freq_at, size, avg_freq_change, run_likelihood, n_transient

In [2]:
scenarios = find_scenarios()
fish_data, clone_data = load_all_scenarios(scenarios)
# clone_names = load_clone_names()

In [3]:
print(f'Found scenarios are: {scenarios}.')

Found scenarios are: ['n1000_r10_y6', 'n500_r15_y6', 'n500_r1_y6', 'n1500_r36_y6', 'n500_r15_y9', 'n1000_r1_y3', 'n1000_r1.5_y6', 'n1000_r9_y6', 'n500_r1_y9', 'n1000_r3_y6_realistic', 'n1500_r36_y9', 'n1000_r4_y6_realistic', 'n1000_r15_y6', 'n1000_r0_y3', 'n1000_r0.5_y6', 'n1000_r8_y6', 'n500_r4_y6', 'n500_r0_y9', 'n1000_r4_y3', 'n1000_r15_y9', 'no_collapse', 'n1000_r5_y6_realistic', 'n1000_r2_y6_realistic', 'n500_r0_y6', 'n500_r4_y9', 'n1000_r36_y3', 'n1500_r4_y9', 'n1500_r0_y6', 'n1000_r8_y6_realistic', 'n1000_r7_y6', 'n1500_r0_y9', 'n1500_r4_y6', 'full_pipeline_trunc', 'n1000_r3_y6', 'n500_r36_y3', 'n1500_r1_y9', 'n1000_r9_y6_realistic', 'n1000_r36_y6_realistic', 'n1000_r2_y6', 'n1500_r15_y3', 'n1500_r1_y6', 'n1000_r15_y6_realistic', 'n1000_r6_y6', 'n1000_r1_y6', 'n1500_r36_y3', 'n500_r1_y3', 'n1000_r1.5_y6_realistic', 'n500_r15_y3', 'n1000_r10_y6_realistic', 'n1000_r5_y6', 'n1000_r1_y9', 'n1000_r0.5_y6_realistic', 'n500_r0_y3', 'n1000_r4_y6', 'n1000_r0_y9', 'n500_r4_y3', 'n1000_r0_

In [4]:
long_dfs = []
for scenario in scenarios:
    path = os.path.join('scenarios', scenario, f'{scenario}_long.csv')
    if os.path.isfile(path):
        long_dfs.append(pd.read_csv(path))

In [5]:
long_results = pd.concat(long_dfs).set_index(['scenario', 'fish', 'clone'])
print(f'Long results has length {len(long_results)}.')
long_results.head()

Long results has length 45065.


init_time  init_freq      r_opt       l_max  \
scenario     fish clone                                                
n1000_r10_y6 0    0            5.0   0.576271   7.778425 -212.832113   
                  1            5.0   0.232345   6.179582 -205.600681   
                  2            5.0   0.057203   9.129557  -71.589854   
                  3            5.0   0.067090  14.911638  -54.953492   
                  4            5.0   0.032486  17.537116 -126.057567   

                            lower      upper       l_null  
scenario     fish clone                                    
n1000_r10_y6 0    0      4.098904  16.090177 -1888.591559  
                  1      3.165454  13.058055  -849.881384  
                  2      4.015566  21.601640  -380.809396  
                  3      4.885854  53.652249  -518.201125  
                  4      8.900827  38.057712 -1569.843414

In [6]:
# Specify order of columns
columns = ['scenario', 'fish', 'clone', 'init_time', 'init_freq', 'last_time',
           'min_time', 'max_time', 'max_freq', 'n_transient', 'size', 'avg_freq_change', 'r_opt',
           'l_max', 'lower', 'upper', 'l_null']

rows = []
for (scenario, fish, clone), data in tqdm(clone_data.items()):
    try:
        if (scenario, fish, clone) in long_results.index:
            row = long_results.loc[scenario, fish, clone].to_dict()
        else:
            row = {col: pd.NA for col in columns}
            it = init_time(data)
            if it is None: continue
            else: row['init_time'] = it

            row['init_freq'] = freq_at(data, row['init_time'])
    
        row['scenario'] = scenario
        row['fish'] = fish
        row['clone'] = clone
        
        row['last_time'] = last_time(data)
        row['min_time'] = min_time(data)
        row['max_time'] = max_time(data)
        row['max_freq'] = freq_at(data, row['max_time'])
        row['n_transient'] = n_transient(data)
        row['size'] = size(data)
            
        row['avg_freq_change'] = avg_freq_change(data)
        
        rows.append(row)

    except Exception as e:
        print(f'Error for scenario={scenario}, fish={fish}, clone={clone}')
        raise

100%|██████████████████████████████████████| 56346/56346 [01:30<00:00, 620.99it/s]


In [7]:
df = pd.DataFrame(rows, columns = columns)
df.to_csv('all_clone_results.csv', index = False)

In [8]:
num_na = len(df[df['r_opt'].isna()])
num_fail = len(df[df['l_max'] < -1e5])

print(f'{num_na} rows have r_opt == NA, and {num_fail} have l_max < -1e5 (indicating a failure to converge).')

101 rows have r_opt == NA, and 0 have l_max < -1e5 (indicating a failure to converge).
